In [1]:
# =============================================================================
# NOTEBOOK 3 — Finetune Gemma 4 with Unsloth (Text Only)
# HawkWatch Gemma Edition | Kaggle Gemma 4 Good Hackathon
#
# WHY TEXT ONLY:
#   Our training data = (scene description TEXT → incident report JSON)
#   No images in training pairs — Gemini already converted frames to text
#   So we finetune language layers only — teaches structured JSON output format
#   Vision layers stay frozen — Gemma already knows how to see images
#
# INPUT FILES (add as Kaggle dataset input before running):
#   hawkwatch_train_balanced.jsonl   (698 training examples)
#   hawkwatch_test.jsonl             (149 test examples)
#
# OUTPUT:
#   Finetuned model pushed to HuggingFace Hub
#
# EXPECTED RUNTIME: ~1.5-2 hours on T4 GPU
# GPU: T4 x1 | Internet: ON
# KAGGLE ACCOUNT: Use Account 2 (saves Account 1 for demo day)
#
# KEY NOTES FROM UNSLOTH DOCS + SAMPLE NOTEBOOK:
#   - Use FastModel (not FastVisionModel) for text-only finetuning
#   - Use tokenizer (not processor) 
#   - Loss of 13-15 at step 1 is NORMAL for E4B — do not panic
#   - train_on_responses_only = only train on JSON output, not user prompt
#   - Use "gemma-4" chat template (not "gemma-4-thinking")
# =============================================================================
 
 
# -----------------------------------------------------------------------------
# CELL 1 — Install Unsloth
# Run this cell then RESTART KERNEL, then run from Cell 2
# -----------------------------------------------------------------------------
# %%capture
# %%capture
import os, re

# Fix huggingface-hub version first
!pip install "huggingface-hub>=1.5.0,<2.0" --upgrade --quiet

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth --upgrade --quiet
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=1.5.0,<2.0" hf_transfer --quiet
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth --quiet

!pip install --no-deps transformers==5.5.0 --quiet
!pip install --no-deps --upgrade timm --quiet

import torch; torch._dynamo.config.recompile_limit = 64
print("Done — RESTART KERNEL NOW then run from Cell 2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 kB 14.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 72.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 13.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 14.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Cell 2
from unsloth import FastModel
import torch

print("Loading Gemma 4 E4B...")

model, tokenizer = FastModel.from_pretrained(
    model_name      = "unsloth/gemma-4-E4B-it",
    dtype           = None,
    max_seq_length  = 2048,
    load_in_4bit    = True,
    full_finetuning = False,
    device_map      = {"": torch.cuda.current_device()},  # ← add this line
)

print("\nModel loaded!")
gpu = torch.cuda.get_device_properties(0)
mem = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"GPU: {gpu.name} | Used: {mem} GB / {round(gpu.total_memory/1024**3,2)} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Gemma 4 E4B...
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]


Model loaded!
GPU: Tesla T4 | Used: 10.15 GB / 14.56 GB


In [3]:
# -----------------------------------------------------------------------------
# CELL 3 — Add LoRA adapters
# -----------------------------------------------------------------------------
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # text-only data — no images
    finetune_language_layers   = True,   # learns our JSON report format
    finetune_attention_modules = True,   # good for structured output
    finetune_mlp_modules       = True,   # always keep on
 
    r            = 16,     # rank 16 suits our 698-example dataset
    lora_alpha   = 16,     # keep equal to r
    lora_dropout = 0,
    bias         = "none",
    random_state = 3407,
)
 
print("LoRA adapters added")
print("Vision layers: FROZEN | Language layers: TRAINABLE")
 
 

LoRA adapters added
Vision layers: FROZEN | Language layers: TRAINABLE


In [4]:
# -----------------------------------------------------------------------------
# CELL 4 — Apply chat template
# -----------------------------------------------------------------------------
from unsloth.chat_templates import get_chat_template
 
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-4",   # not "gemma-4-thinking" — direct answers
)
 
print("Chat template: gemma-4 applied")

Chat template: gemma-4 applied


In [5]:
import json
from pathlib import Path
 
# Auto-find dataset files
def find_file(filename):
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]
    return None
 
train_path = find_file("hawkwatch_train_balanced.jsonl")
test_path  = find_file("hawkwatch_test.jsonl")
 
if not train_path:
    print("ERROR: hawkwatch_train_balanced.jsonl not found")
    print("Add your dataset as Kaggle input before running")
else:
    print(f"Train: {train_path}")
    print(f"Test:  {test_path}")
 
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]
 
train_raw = load_jsonl(train_path)
test_raw  = load_jsonl(test_path)
 
print(f"\nLoaded {len(train_raw)} train, {len(test_raw)} test examples")
print(f"\nSample:")
print(f"  Input:  {train_raw[0]['input'][:100]}...")
print(f"  Output: {train_raw[0]['output'][:100]}...")
 
 

Train: /kaggle/input/datasets/dj4242/dataset/hawkwatch_train_balanced.jsonl
Test:  /kaggle/input/datasets/dj4242/dataset/hawkwatch_test.jsonl

Loaded 698 train, 149 test examples

Sample:
  Input:  The surveillance frame shows an empty residential living room with a television, a red armchair, and...
  Output: {
  "scene_description": "An empty residential living room containing a television, a red armchair, ...


In [6]:
# -----------------------------------------------------------------------------
# CELL 6 — Convert to conversation format and apply chat template
# -----------------------------------------------------------------------------
# Format used by Unsloth text finetuning (from official text notebook):
# conversations = [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]
# Then apply_chat_template converts to flat text with <|turn> tokens
 
def convert_to_conversation(sample: dict) -> dict:
    """Convert Alpaca format to conversation format for Unsloth."""
    user_text = f"{sample['instruction']}\n\nScene: {sample['input']}"
    return {
        "conversations": [
            {"role": "user",      "content": user_text},
            {"role": "assistant", "content": sample["output"]},
        ]
    }
 
# Convert all examples
train_convs = [convert_to_conversation(s) for s in train_raw]
test_convs  = [convert_to_conversation(s) for s in test_raw]
 
# Apply chat template — converts to flat text string with Gemma 4 tokens
# removeprefix('<bos>') because trainer adds it — avoid duplicate
def formatting_prompts_func(examples):
    texts = []
    for convo in examples["conversations"]:
        text = tokenizer.apply_chat_template(
            convo,
            tokenize              = False,
            add_generation_prompt = False,
        ).removeprefix("<bos>")
        texts.append(text)
    return {"text": texts}
 
from datasets import Dataset
 
train_dataset = Dataset.from_list(train_convs)
test_dataset  = Dataset.from_list(test_convs)
 
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
test_dataset  = test_dataset.map(formatting_prompts_func,  batched=True)
 
print(f"Dataset formatted")
print(f"\nSample formatted text (first 300 chars):")
print(train_dataset[0]["text"][:300])
 

Map:   0%|          | 0/698 [00:00<?, ? examples/s]

Map:   0%|          | 0/149 [00:00<?, ? examples/s]

Dataset formatted

Sample formatted text (first 300 chars):
<|turn>user
You are a security monitoring AI. Analyze this surveillance scene description and generate a structured incident report as JSON.

Scene: The surveillance frame shows an empty residential living room with a television, a red armchair, and a rug. There are no people present in the room, an


In [7]:
print("BASE MODEL output (before finetuning)")
print("Save this — compare with Cell 11 after training\n")

SAMPLE_INPUT    = train_raw[0]["input"]
SAMPLE_EXPECTED = train_raw[0]["output"]

# Correct format — content must be a LIST of dicts, not a plain string
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": f"You are a security monitoring AI. Generate a structured incident report as JSON.\n\nScene: {SAMPLE_INPUT}"
            }
        ]
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize              = True,
    add_generation_prompt = True,
    return_dict           = True,
    return_tensors        = "pt",
).to("cuda")

from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True)

print(f"Scene: {SAMPLE_INPUT[:120]}...\n")
print("--- BASE MODEL OUTPUT ---")
_ = model.generate(
    **inputs,
    streamer       = streamer,
    max_new_tokens = 300,
    use_cache      = True,
    temperature    = 1.0,
    top_p          = 0.95,
    top_k          = 64,
)
print("\n--- EXPECTED OUTPUT ---")
print(SAMPLE_EXPECTED[:300])
print('dj')

BASE MODEL output (before finetuning)
Save this — compare with Cell 11 after training

Scene: The surveillance frame shows an empty residential living room with a television, a red armchair, and a rug. There are no...

--- BASE MODEL OUTPUT ---
```json
{
  "incident_report": {
    "report_id": "IR-20231027-001",
    "timestamp_generated": "2023-10-27T10:30:00Z",
    "monitoring_system": "SurveillanceAI_v3.1",
    "scene_details": {
      "location_type": "Residential Interior",
      "description": "Living room",
      "objects_present": [
        "Television",
        "Red Armchair",
        "Rug"
      ],
      "occupancy_status": "Empty"
    },
    "detection_summary": {
      "activity_detected": false,
      "anomaly_score": 0.0,
      "threat_level": "Low",
      "analysis": "No unusual or concerning activity detected in the monitored frame."
    },
    "incident_classification": {
      "type": "Normal Operation / No Incident",
      "severity": "Informational"
    },
    "recom

In [8]:
# -----------------------------------------------------------------------------
# CELL 8 — Setup trainer with train_on_responses_only
# -----------------------------------------------------------------------------
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
 
steps_per_epoch = len(train_dataset) // 4  # effective batch = 4
total_steps     = steps_per_epoch * 2       # 2 epochs
 
print(f"Training config:")
print(f"  Examples:        {len(train_dataset)}")
print(f"  Epochs:          2")
print(f"  Steps/epoch:     {steps_per_epoch}")
print(f"  Total steps:     {total_steps}")
print(f"  Effective batch: 4 (batch=1 x grad_accum=4)")
 
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = test_dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        max_grad_norm               = 0.3,
        warmup_steps                = 10,
        num_train_epochs            = 2,
        learning_rate               = 2e-4,
        logging_steps               = 5,
        save_strategy               = "steps",
        save_steps                  = 50,       # checkpoint every 50 steps
        save_total_limit            = 3,        # keep last 3 checkpoints
        eval_strategy               = "steps",
        eval_steps                  = 50,       # evaluate every 50 steps
        optim                       = "adamw_8bit",
        weight_decay                = 0.001,
        lr_scheduler_type           = "cosine",
        seed                        = 3407,
        output_dir                  = "/kaggle/working/hawkwatch-gemma4",
        report_to                   = "none",
    ),
)
 
# CRITICAL — only train on assistant JSON output, not user instruction
# This makes training much more efficient and focused
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part    = "<|turn>model\n",
)
 
print("\nTrainer ready with train_on_responses_only enabled")
print("Model only learns from JSON outputs, not user instructions")
 
 

Training config:
  Examples:        698
  Epochs:          2
  Steps/epoch:     174
  Total steps:     348
  Effective batch: 4 (batch=1 x grad_accum=4)


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/698 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/149 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/698 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/698 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/149 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/149 [00:00<?, ? examples/s]


Trainer ready with train_on_responses_only enabled
Model only learns from JSON outputs, not user instructions


In [9]:
# -----------------------------------------------------------------------------
# CELL 8b — Check for existing checkpoint (for resume support)
# -----------------------------------------------------------------------------
from pathlib import Path
 
CHECKPOINT_DIR = Path("/kaggle/working/hawkwatch-gemma4")
checkpoint = None
 
if CHECKPOINT_DIR.exists():
    checkpoints = sorted([
        d for d in CHECKPOINT_DIR.iterdir()
        if d.is_dir() and "checkpoint" in d.name
    ])
    if checkpoints:
        checkpoint = str(checkpoints[-1])
        print(f"Found checkpoint: {checkpoint}")
        print("Will RESUME from this checkpoint")
    else:
        print("No checkpoint found — starting fresh")
else:
    print("Starting fresh training")
 

No checkpoint found — starting fresh


In [10]:
# -----------------------------------------------------------------------------
# CELL 9 — TRAIN
# -----------------------------------------------------------------------------
print("Starting training...")
print("=" * 50)
print("EXPECTED LOSS PROGRESSION:")
print("  Steps 1-5:   13-16  (NORMAL for E4B — do not stop)")
print("  Steps 6-15:  drops to 5-7")
print("  Steps 15-30: drops to 2-3")
print("  Steps 30+:   stabilizes 1-2")
print("=" * 50)
print()
 
mem_before = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"GPU memory before: {mem_before} GB\n")
 
if checkpoint:
    print(f"Resuming from: {checkpoint}\n")
    trainer_stats = trainer.train(resume_from_checkpoint=checkpoint)
else:
    trainer_stats = trainer.train()
 
print(f"\n{'='*50}")
print(f"TRAINING COMPLETE")
print(f"  Final loss:    {trainer_stats.training_loss:.4f}")
print(f"  Total steps:   {trainer_stats.global_step}")
print(f"  Training time: {trainer_stats.metrics['train_runtime']/60:.1f} minutes")
mem_after = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"  Peak GPU mem:  {mem_after} GB")
print(f"{'='*50}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Starting training...
EXPECTED LOSS PROGRESSION:
  Steps 1-5:   13-16  (NORMAL for E4B — do not stop)
  Steps 6-15:  drops to 5-7
  Steps 15-30: drops to 2-3
  Steps 30+:   stabilizes 1-2

GPU memory before: 10.34 GB



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 698 | Num Epochs = 2 | Total steps = 350
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 36,700,160 of 8,032,856,608 (0.46% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss,Validation Loss
50,0.869345,1.537502
100,0.496254,1.404889
150,0.408437,1.330022
200,0.415535,1.374203
250,0.309688,1.393797
300,0.372591,1.376339
350,0.279073,1.374646


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/hawkwatch-gemma4/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/hawkwatch-gemma4/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/hawkwatch-gemma4/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/hawkwatch-gemma4/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/hawkwatch-gemma4/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/hawkwatch-gemma4/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/hawkwatch-gemma4/checkpoint-350/tokenizer_config.json.



TRAINING COMPLETE
  Final loss:    0.9073
  Total steps:   350
  Training time: 28.8 minutes
  Peak GPU mem:  12.35 GB


In [22]:
# -----------------------------------------------------------------------------
# CELL 10 — Save locally
# -----------------------------------------------------------------------------
LOCAL_PATH = "/kaggle/working/hawkwatch-gemma4-lora"
print(f"Saving LoRA adapter to {LOCAL_PATH}...")
 
model.save_pretrained(LOCAL_PATH)
tokenizer.save_pretrained(LOCAL_PATH)
 
import os
print("\nSaved files:")
for f in sorted(os.listdir(LOCAL_PATH)):
    size = os.path.getsize(f"{LOCAL_PATH}/{f}") / 1e6
    print(f"  {f} ({size:.1f} MB)")

Saving LoRA adapter to /kaggle/working/hawkwatch-gemma4-lora...


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/hawkwatch-gemma4-lora/tokenizer_config.json.



Saved files:
  README.md (0.0 MB)
  adapter_config.json (0.0 MB)
  adapter_model.safetensors (146.9 MB)
  chat_template.jinja (0.0 MB)
  processor_config.json (0.0 MB)
  tokenizer.json (32.2 MB)
  tokenizer_config.json (0.0 MB)


In [24]:
# -----------------------------------------------------------------------------
# CELL 11 — Test FINETUNED model (compare with Cell 7)
# -----------------------------------------------------------------------------
print("FINETUNED MODEL output (compare with Cell 7)\n")

FastModel.for_inference(model)

# Use EXACT same instruction as training data
INSTRUCTION = train_raw[0]["instruction"]  # pulls exact instruction used in training
SAMPLE_INPUT = train_raw[0]["input"]
SAMPLE_EXPECTED = train_raw[0]["output"]

# Add field reminder to guide output format
FIELD_REMINDER = """
Return ONLY a JSON object with these exact fields:
{
  "scene_description": "...",
  "activity_detected": "...",
  "persons_count": 0,
  "severity": "CRITICAL or WARNING or CLEAR",
  "category": "Crime or Medical Emergency or Suspicious Activity or Disaster or Normal",
  "confidence": 0-100,
  "recommended_action": "...",
  "objects_of_interest": []
}"""

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": f"{INSTRUCTION}{FIELD_REMINDER}\n\nScene: {SAMPLE_INPUT}"
            }
        ]
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize              = True,
    add_generation_prompt = True,
    return_dict           = True,
    return_tensors        = "pt",
).to("cuda")

print(f"Scene: {SAMPLE_INPUT[:120]}...\n")
print("--- FINETUNED MODEL OUTPUT ---")
_ = model.generate(
    **inputs,
    streamer       = streamer,
    max_new_tokens = 400,
    use_cache      = True,
    temperature    = 1.0,
    top_p          = 0.95,
    top_k          = 64,
)
print("\n--- EXPECTED OUTPUT ---")
print(SAMPLE_EXPECTED)

FINETUNED MODEL output (compare with Cell 7)

Scene: The surveillance frame shows an empty residential living room with a television, a red armchair, and a rug. There are no...

--- FINETUNED MODEL OUTPUT ---
```json
{
  "scene_description": "An empty residential living room featuring a television, a red armchair, and a rug. The room is tidy and undisturbed.",
  "activity_detected": "No activity detected; the scene is clear.",
  "persons_count": 0,
  "severity": "CLEAR",
  "category": "Normal",
  "confidence": 100,
  "recommended_action": "Continue routine monitoring of the property.",
  "objects_of_interest": [
    "television",
    "red armchair",
    "rug"
  ]
}
```<turn|>

--- EXPECTED OUTPUT ---
{
  "scene_description": "An empty residential living room containing a television, a red armchair, and a rug. The environment is stable with no signs of disturbance.",
  "activity_detected": "No activity detected; the room is vacant.",
  "persons_count": 0,
  "severity": "CLEAR",
  "categ

In [25]:
# -----------------------------------------------------------------------------
# CELL 12 — Evaluation on full test set (149 samples)
# -----------------------------------------------------------------------------
import json, re
 
def parse_json(text):
    try:
        return json.loads(text.strip())
    except:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            try:
                return json.loads(m.group())
            except:
                pass
    return None
 
print("Running evaluation on full test set...")
print(f"Total test samples: {len(test_raw)}\n")
 
results = {
    "total":            len(test_raw),
    "valid_json":       0,
    "all_fields":       0,
    "severity_correct": 0,
    "category_correct": 0,
}
 
REQUIRED_FIELDS = [
    "scene_description", "activity_detected", "persons_count",
    "severity", "category", "confidence",
    "recommended_action", "objects_of_interest"
]
 
for i, sample in enumerate(test_raw):
    if i % 10 == 0:
        print(f"  Progress: {i}/{len(test_raw)}")
 
    expected = json.loads(sample["output"])
 
    FIELD_REMINDER = """
Return ONLY a JSON object with these exact fields:
{
  "scene_description": "...",
  "activity_detected": "...",
  "persons_count": 0,
  "severity": "CRITICAL or WARNING or CLEAR",
  "category": "Crime or Medical Emergency or Suspicious Activity or Disaster or Normal",
  "confidence": 0-100,
  "recommended_action": "...",
  "objects_of_interest": []
}"""

    msgs = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": f"{sample['instruction']}{FIELD_REMINDER}\n\nScene: {sample['input']}"
            }
        ]
    }
]
    inps = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    ).to("cuda")
 
    with torch.no_grad():
        out = model.generate(**inps, max_new_tokens=350, use_cache=True,
                             temperature=1.0, top_p=0.95, top_k=64)
 
    decoded = tokenizer.decode(out[0][inps["input_ids"].shape[1]:], skip_special_tokens=True)
    parsed  = parse_json(decoded)
 
    if parsed:
        results["valid_json"] += 1
 
        # Check all fields present
        missing = [f for f in REQUIRED_FIELDS if f not in parsed]
        if not missing:
            results["all_fields"] += 1
 
        # Severity accuracy
        if parsed.get("severity") == expected.get("severity"):
            results["severity_correct"] += 1
 
        # Category accuracy
        if parsed.get("category") == expected.get("category"):
            results["category_correct"] += 1
 
n = results["total"]
print(f"\n{'='*50}")
print(f"EVALUATION RESULTS (vs Gemini-labeled ground truth)")
print(f"{'='*50}")
print(f"Total test samples:    {n}")
print(f"Valid JSON output:     {results['valid_json']}/{n} ({results['valid_json']/n*100:.1f}%)")
print(f"All fields present:    {results['all_fields']}/{n} ({results['all_fields']/n*100:.1f}%)")
print(f"Severity accuracy:     {results['severity_correct']}/{n} ({results['severity_correct']/n*100:.1f}%)")
print(f"Category accuracy:     {results['category_correct']}/{n} ({results['category_correct']/n*100:.1f}%)")
print(f"{'='*50}")
print("\nSave these numbers — they go in your submission writeup")
print("Run Notebook 1 Cell 9 (base model) on same test set to show improvement")
 
# Save results
import json
with open("/kaggle/working/eval_results_finetuned.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nSaved: /kaggle/working/eval_results_finetuned.json")
 

Running evaluation on full test set...
Total test samples: 149

  Progress: 0/149
  Progress: 10/149
  Progress: 20/149
  Progress: 30/149
  Progress: 40/149
  Progress: 50/149
  Progress: 60/149
  Progress: 70/149
  Progress: 80/149
  Progress: 90/149
  Progress: 100/149
  Progress: 110/149
  Progress: 120/149
  Progress: 130/149
  Progress: 140/149

EVALUATION RESULTS (vs Gemini-labeled ground truth)
Total test samples:    149
Valid JSON output:     149/149 (100.0%)
All fields present:    149/149 (100.0%)
Severity accuracy:     98/149 (65.8%)
Category accuracy:     99/149 (66.4%)

Save these numbers — they go in your submission writeup
Run Notebook 1 Cell 9 (base model) on same test set to show improvement

Saved: /kaggle/working/eval_results_finetuned.json


In [26]:
# # Eval on BASE model (before finetuning)
# # We need to reload base model without LoRA for fair comparison

# print("Loading BASE model for comparison eval...")

# from unsloth import FastModel
# base_model, base_tokenizer = FastModel.from_pretrained(
#     model_name      = "unsloth/gemma-4-E4B-it",
#     dtype           = None,
#     max_seq_length  = 2048,
#     load_in_4bit    = True,
#     full_finetuning = False,
#     device_map      = {"": torch.cuda.current_device()},
# )

# from unsloth.chat_templates import get_chat_template
# base_tokenizer = get_chat_template(base_tokenizer, chat_template="gemma-4")

# FastModel.for_inference(base_model)

# FIELD_REMINDER = """
# Return ONLY a JSON object with these exact fields:
# {
#   "scene_description": "...",
#   "activity_detected": "...",
#   "persons_count": 0,
#   "severity": "CRITICAL or WARNING or CLEAR",
#   "category": "Crime or Medical Emergency or Suspicious Activity or Disaster or Normal",
#   "confidence": 0-100,
#   "recommended_action": "...",
#   "objects_of_interest": []
# }"""

# print("Running base model evaluation on 149 test samples...")

# base_results = {
#     "total":            len(test_raw),
#     "valid_json":       0,
#     "all_fields":       0,
#     "severity_correct": 0,
#     "category_correct": 0,
# }

# for i, sample in enumerate(test_raw):
#     if i % 10 == 0:
#         print(f"  Progress: {i}/{len(test_raw)}")

#     expected = json.loads(sample["output"])

#     msgs = [
#         {
#             "role": "user",
#             "content": [
#                 {
#                     "type": "text",
#                     "text": f"{sample['instruction']}{FIELD_REMINDER}\n\nScene: {sample['input']}"
#                 }
#             ]
#         }
#     ]

#     inps = base_tokenizer.apply_chat_template(
#         msgs, tokenize=True, add_generation_prompt=True,
#         return_dict=True, return_tensors="pt"
#     ).to("cuda")

#     with torch.no_grad():
#         out = base_model.generate(
#             **inps, max_new_tokens=350, use_cache=True,
#             temperature=1.0, top_p=0.95, top_k=64
#         )

#     decoded = base_tokenizer.decode(
#         out[0][inps["input_ids"].shape[1]:],
#         skip_special_tokens=True
#     )
#     parsed = parse_json(decoded)

#     if parsed:
#         base_results["valid_json"] += 1
#         missing = [f for f in REQUIRED_FIELDS if f not in parsed]
#         if not missing:
#             base_results["all_fields"] += 1
#         if parsed.get("severity") == expected.get("severity"):
#             base_results["severity_correct"] += 1
#         if parsed.get("category") == expected.get("category"):
#             base_results["category_correct"] += 1

# n = base_results["total"]
# print(f"\n{'='*50}")
# print(f"BASE MODEL RESULTS")
# print(f"{'='*50}")
# print(f"Valid JSON:         {base_results['valid_json']}/{n} ({base_results['valid_json']/n*100:.1f}%)")
# print(f"All fields present: {base_results['all_fields']}/{n} ({base_results['all_fields']/n*100:.1f}%)")
# print(f"Severity accuracy:  {base_results['severity_correct']}/{n} ({base_results['severity_correct']/n*100:.1f}%)")
# print(f"Category accuracy:  {base_results['category_correct']}/{n} ({base_results['category_correct']/n*100:.1f}%)")

# print(f"\n{'='*50}")
# print(f"COMPARISON TABLE (for submission writeup)")
# print(f"{'='*50}")
# print(f"{'Metric':<25} {'Base Gemma 4':>15} {'Finetuned (Ours)':>18} {'Improvement':>12}")
# print(f"{'-'*70}")

# metrics = [
#     ("Valid JSON",          base_results['valid_json'],       149),
#     ("All Fields Present",  base_results['all_fields'],       149),
#     ("Severity Accuracy",   base_results['severity_correct'], 98),
#     ("Category Accuracy",   base_results['category_correct'], 99),
# ]

# for name, base_val, ft_val in metrics:
#     base_pct = base_val / n * 100
#     ft_pct   = ft_val  / n * 100
#     diff     = ft_pct - base_pct
#     arrow    = "↑" if diff > 0 else "↓"
#     print(f"{name:<25} {base_pct:>14.1f}% {ft_pct:>17.1f}% {arrow}{abs(diff):>10.1f}%")

# # Save comparison
# comparison = {"base": base_results, "finetuned": results}
# with open("/kaggle/working/eval_comparison.json", "w") as f:
#     json.dump(comparison, f, indent=2)
# print(f"\nSaved: /kaggle/working/eval_comparison.json")

Loading BASE model for comparison eval...
==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


OutOfMemoryError: CUDA out of memory. Tried to allocate 9.82 GiB. GPU 0 has a total capacity of 14.56 GiB of which 3.93 GiB is free. Including non-PyTorch memory, this process has 10.63 GiB memory in use. Of the allocated memory 10.32 GiB is allocated by PyTorch, and 152.70 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# -----------------------------------------------------------------------------
# CELL 13 — Push to HuggingFace
# -----------------------------------------------------------------------------
HF_TOKEN    = ""       # huggingface.co/settings/tokens (Write)
HF_USERNAME = "Dharya"
MODEL_NAME  = "secure_sight-gemma4-security"
REPO_ID     = f"{HF_USERNAME}/{MODEL_NAME}"
 
print(f"Pushing to HuggingFace: {REPO_ID}")
 
model.push_to_hub(REPO_ID, token=HF_TOKEN)
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)
 
print(f"\nDone! Model live at: https://huggingface.co/{REPO_ID}")
print(f"\nUpdate your project .env:")
print(f"  HUGGINGFACE_MODEL_ID={REPO_ID}")
print(f"\n=== NOTEBOOK 3 COMPLETE ===")
print(f"Next: Notebook 4 — load finetuned model, expose via ngrok")

Pushing to HuggingFace: Dharya/secure_sight-gemma4-security


README.md:   0%|          | 0.00/568 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/Dharya/secure_sight-gemma4-security


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpiae4z3f_/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Done! Model live at: https://huggingface.co/Dharya/secure_sight-gemma4-security

Update your project .env:
  HUGGINGFACE_MODEL_ID=Dharya/secure_sight-gemma4-security

=== NOTEBOOK 3 COMPLETE ===
Next: Notebook 4 — load finetuned model, expose via ngrok


In [1]:
from pathlib import Path

checkpoint_dir = Path("/kaggle/working/hawkwatch-gemma4")
if checkpoint_dir.exists():
    checkpoints = sorted([d for d in checkpoint_dir.iterdir() if "checkpoint" in d.name])
    print(f"Found {len(checkpoints)} checkpoints:")
    for c in checkpoints:
        print(f"  {c.name}")
else:
    print("Working directory cleared — need to retrain")


Working directory cleared — need to retrain
